In [8]:
!ls ../input/notebooks/nightfury1103/judge-rationales/model_path/selected_heuristic_1

config.json		 spiece.model		training_args.bin
pytorch_model.bin	 tokenizer_config.json
special_tokens_map.json  tokenizer.json


In [9]:
import os, sys
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import T5ForConditionalGeneration, AutoTokenizer
from datasets import load_dataset

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


In [12]:
TYPE = 'esnli'

datasets = load_dataset('csv', data_files={'anli': "/kaggle/input/datasets/nightfury1103/anli-esnli-test/anli_test.csv",'esnli': "/kaggle/input/datasets/nightfury1103/anli-esnli-test/esnli_test.csv", 'cqa': "/kaggle/input/datasets/nightfury1103/anli-esnli-test/cqa.csv"})


Generating anli split: 0 examples [00:00, ? examples/s]

Generating esnli split: 0 examples [00:00, ? examples/s]

Generating cqa split: 0 examples [00:00, ? examples/s]

In [13]:
test_dataset = datasets[TYPE]
df = pd.DataFrame(test_dataset)
test_inputs = df['input'].tolist()


In [17]:
def get_pred(inputs, model, tokenizer, batch_size=64, max_input_length=1024, max_new_tokens=64):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model.to(device)
    model.eval()
    model.config.use_cache = True

    use_autocast = device == 'cuda'
    if device == 'cuda' and hasattr(torch.cuda, 'is_bf16_supported') and torch.cuda.is_bf16_supported():
        autocast_dtype = torch.bfloat16
    elif device == 'cuda':
        autocast_dtype = torch.float16
    else:
        autocast_dtype = None

    predictions = []
    for start in tqdm(range(0, len(inputs), batch_size), desc='Generating'):
        batch_inputs = inputs[start:start + batch_size]
        tokenized = tokenizer(
            batch_inputs,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_input_length,
        )
        tokenized = {k: v.to(device) for k, v in tokenized.items()}

        with torch.inference_mode():
            if use_autocast:
                with torch.autocast(device_type='cuda', dtype=autocast_dtype):
                    output = model.generate(**tokenized, max_new_tokens=max_new_tokens)
            else:
                output = model.generate(**tokenized, max_new_tokens=max_new_tokens)

        decoded = tokenizer.batch_decode(output, skip_special_tokens=True)
        predictions.extend(text.strip() for text in decoded)

    return predictions


In [16]:
PATH = '../input/notebooks/nightfury1103/judge-rationales/model_path/'
filename = 'selected_heuristic_1'
BATCH_SIZE = 64
MAX_INPUT_LENGTH = 1024
MAX_NEW_TOKENS = 64
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model = T5ForConditionalGeneration.from_pretrained(f'{PATH}/{filename}').to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(f'{PATH}/{filename}')

df = df.copy()
df[filename] = get_pred(
    test_inputs,
    model,
    tokenizer,
    batch_size=BATCH_SIZE,
    max_input_length=MAX_INPUT_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS,
)

pred_norm = df[filename].astype(str).str.strip().str.lower()
label_norm = df['label'].astype(str).str.strip().str.lower()

score_1 = (pred_norm == label_norm).mean()
score_2 = sum(label in output for label, output in zip(label_norm, pred_norm)) / len(df)
score_3 = sum(label in output.lower() for label, output in zip(label_norm, df[filename].astype(str))) / len(df)

print(f'name: {filename}, acc1: {score_1}, acc2: {score_2}, acc3: {score_3}')
df[['input', 'label', filename]].head()


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/9824 [00:00<?, ? examples/s]

KeyboardInterrupt: 